# Problem Statement 3: Robot Strategy Battle

### Artificial Intelligence Group Assignment

This notebook implements a depth-limited Minimax algorithm and
Minimax with Alpha-Beta pruning for the Robot Strategy Battle problem.

The performance of the two approaches is also compared.

## 1. Problem Description

In this problem, two robots compete on a warehouse grid to collect
energy cells.

- Robot A is the MAX player.
- Robot B is the MIN player.
- Each energy cell gives 10 points.
- Robots can move Up, Right, Down, or Left.
- Robots cannot move through obstacles.
- The robots take alternate turns.

MAX tries to maximize the difference between its score and MIN's score,
while MIN tries to minimize it.

Since searching the complete game tree can become expensive, a
depth-limited search is used.

## 2. Algorithms Used

Two search algorithms are implemented:

### Minimax

Minimax explores the game tree up to a given depth.

- MAX chooses the move with the highest evaluation.
- MIN chooses the move with the lowest evaluation.

### Alpha-Beta Pruning

Alpha-Beta pruning improves Minimax by avoiding branches that cannot
affect the final decision.

Two versions of Alpha-Beta are considered:

1. Normal move ordering: UP → RIGHT → DOWN → LEFT
2. Heuristic move ordering: moves that collect an energy cell are
   considered first.

The results of these approaches are compared using the number of
nodes generated, nodes expanded, nodes pruned, and execution time.

In [13]:
import time

moves = [
    (-1, 0, "UP"),
    (0, 1, "RIGHT"),
    (1, 0, "DOWN"),
    (0, -1, "LEFT")
]

board = []
r = 0
c = 0
d = 0

a = None
b = None
e = set()

ng = 0
ne = 0
np = 0

In [14]:
r, c = map(int, input().split())

board = []

for i in range(r):
    s = input()
    board.append(s)

    for j in range(c):
        if s[j] == 'A':
            a = (i, j)
        elif s[j] == 'B':
            b = (i, j)
        elif s[j] == 'E':
            e.add((i, j))

player = input()
d = int(input())

state = (a, b, e, player, 0, 0)

5 7
A.E.#..
.#..#E.
...#...
#E#..#.
....E.B
MAX
5


## 3. State Representation

A game state contains the information needed to continue the search.

It is represented as:

(A position, B position, remaining energy cells, current player, MAX score, MIN score)

For example, the initial state contains the positions of both robots,
all available energy cells, and both scores set to zero.

Whenever a robot moves onto an uncollected energy cell, that cell is
removed from the state and the corresponding score increases by 10.

In [15]:
def valid(x, y):
    return 0 <= x < r and 0 <= y < c and board[x][y] != '#'


def get_moves(pos):
    x, y = pos
    ans = []

    for dx, dy, name in moves:
        nx = x + dx
        ny = y + dy

        if valid(nx, ny):
            ans.append(((nx, ny), name))

    return ans


def make_state(state, pos, player):
    ap, bp, en, turn, sa, sb = state

    if player == "MAX":
        ap = pos

        if pos in en:
            en = en.copy()
            en.remove(pos)
            sa += 10

        turn = "MIN"

    else:
        bp = pos

        if pos in en:
            en = en.copy()
            en.remove(pos)
            sb += 10

        turn = "MAX"

    return (ap, bp, en, turn, sa, sb)

## 4. Evaluation Function

The evaluation function is used when the search reaches the depth limit
or a terminal state.

The evaluation is:

Evaluation = MAX Score - MIN Score + positional advantage

The positional advantage is calculated using the nearest remaining
energy cell.

Positional Advantage =
Distance(B, nearest remaining E) - Distance(A, nearest remaining E)

Manhattan distance is used:

Distance = |r1 - r2| + |c1 - c2|

If there are no remaining energy cells, the positional advantage is 0.

A higher evaluation value is better for MAX.

In [16]:
def evaluate(state):
    ap, bp, en, turn, sa, sb = state

    if len(en) == 0:
        return sa - sb

    da = float('inf')
    db = float('inf')

    for x, y in en:
        aa = abs(ap[0] - x) + abs(ap[1] - y)
        bb = abs(bp[0] - x) + abs(bp[1] - y)

        da = min(da, aa)
        db = min(db, bb)

    p = db - da

    return sa - sb + p

## 5. Minimax

The Minimax function recursively explores the game tree.

At MAX's turn, the maximum child value is selected.

At MIN's turn, the minimum child value is selected.

The search stops when the required depth is reached. At that point,
the evaluation function is used.

In [17]:
def minimax(state, dep, player):
    global ng, ne

    if dep == 0:
        ne += 1
        return evaluate(state)

    if player == "MAX":
        ms = get_moves(state[0])

        if len(ms) == 0:
            ne += 1
            return evaluate(state)

        best = -float('inf')

        for pos, name in ms:
            ng += 1

            ns = make_state(state, pos, "MAX")
            val = minimax(ns, dep - 1, "MIN")

            best = max(best, val)

        return best

    else:
        ms = get_moves(state[1])

        if len(ms) == 0:
            ne += 1
            return evaluate(state)

        best = float('inf')

        for pos, name in ms:
            ng += 1

            ns = make_state(state, pos, "MIN")
            val = minimax(ns, dep - 1, "MAX")

            best = min(best, val)

        return best

In [18]:
def find_minimax(state):
    global ng

    best_val = -float('inf')
    best_move = None

    for pos, name in get_moves(state[0]):
        ng += 1

        ns = make_state(state, pos, "MAX")
        val = minimax(ns, d - 1, "MIN")

        if val > best_val:
            best_val = val
            best_move = name

    return best_move, best_val

## 6. Alpha-Beta Pruning

Alpha-Beta pruning gives the same Minimax result while potentially
examining fewer nodes.

Alpha represents the best value currently guaranteed for MAX.

Beta represents the best value currently guaranteed for MIN.

When alpha becomes greater than or equal to beta, the remaining
branches do not need to be explored.

In [19]:
def alpha_beta(state, dep, player, al, be, order):
    global ng, ne, np

    if dep == 0:
        ne += 1
        return evaluate(state)

    if player == "MAX":
        ms = get_moves(state[0])

        if len(ms) == 0:
            ne += 1
            return evaluate(state)

        if order:
            ms.sort(
                key=lambda x: 1 if x[0] in state[2] else 0,
                reverse=True
            )

        best = -float('inf')

        for pos, name in ms:
            ng += 1

            ns = make_state(state, pos, "MAX")
            val = alpha_beta(ns, dep - 1, "MIN", al, be, order)

            best = max(best, val)
            al = max(al, best)

            if al >= be:
                np += 1
                break

        return best

    else:
        ms = get_moves(state[1])

        if len(ms) == 0:
            ne += 1
            return evaluate(state)

        if order:
            ms.sort(
                key=lambda x: 1 if x[0] in state[2] else 0,
                reverse=True
            )

        best = float('inf')

        for pos, name in ms:
            ng += 1

            ns = make_state(state, pos, "MIN")
            val = alpha_beta(ns, dep - 1, "MAX", al, be, order)

            best = min(best, val)
            be = min(be, best)

            if al >= be:
                np += 1
                break

        return best

In [20]:
def find_alpha_beta(state, order):
    global ng

    best_val = -float('inf')
    best_move = None

    ms = get_moves(state[0])

    if order:
        ms.sort(
            key=lambda x: 1 if x[0] in state[2] else 0,
            reverse=True
        )

    for pos, name in ms:
        ng += 1

        ns = make_state(state, pos, "MAX")

        val = alpha_beta(
            ns,
            d - 1,
            "MIN",
            -float('inf'),
            float('inf'),
            order
        )

        if val > best_val:
            best_val = val
            best_move = name

    return best_move, best_val

## 7. Running Minimax

The initial state is passed to Minimax and the execution time is
recorded.

In [21]:
ng = 0
ne = 0

t = time.time()

mm_move, mm_val = find_minimax(state)

mm_time = time.time() - t

mm_ng = ng
mm_ne = ne

print("Algorithm: Minimax")
print("Best Move:", mm_move)
print("Evaluation Score =", mm_val)
print("Nodes Generated =", ng)
print("Nodes Expanded =", ne)
print("Search Depth =", d)
print("Execution Time =", mm_time)

Algorithm: Minimax
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 36
Search Depth = 5
Execution Time = 0.00027179718017578125


## 8. Running Alpha-Beta

First, Alpha-Beta is tested using the normal move order:

UP → RIGHT → DOWN → LEFT

In [22]:
ng = 0
ne = 0
np = 0

t = time.time()

ab_move, ab_val = find_alpha_beta(state, False)

ab_time = time.time() - t

ab_ng = ng
ab_ne = ne
ab_np = np

print("Algorithm: Alpha-Beta")
print("Best Move:", ab_move)
print("Evaluation Score =", ab_val)
print("Nodes Generated =", ng)
print("Nodes Expanded =", ne)
print("Nodes Pruned =", np)
print("Search Depth =", d)
print("Execution Time =", ab_time)

Algorithm: Alpha-Beta
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 51
Nodes Expanded = 24
Nodes Pruned = 9
Search Depth = 5
Execution Time = 0.0002574920654296875


## 9. Alpha-Beta with Move Ordering

In this version, moves that immediately collect an energy cell are
considered first.

The purpose of move ordering is to obtain good alpha and beta bounds
earlier, which can increase the amount of pruning.

In [23]:
ng = 0
ne = 0
np = 0

t = time.time()

ab2_move, ab2_val = find_alpha_beta(state, True)

ab2_time = time.time() - t

ab2_ng = ng
ab2_ne = ne
ab2_np = np

print("Algorithm: Alpha-Beta with Move Ordering")
print("Best Move:", ab2_move)
print("Evaluation Score =", ab2_val)
print("Nodes Generated =", ng)
print("Nodes Expanded =", ne)
print("Nodes Pruned =", np)
print("Search Depth =", d)
print("Execution Time =", ab2_time)

Algorithm: Alpha-Beta with Move Ordering
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 48
Nodes Expanded = 22
Nodes Pruned = 8
Search Depth = 5
Execution Time = 0.0002665519714355469


## 10. Comparison

The three approaches can be compared using the number of nodes
generated, nodes expanded, nodes pruned, and execution time.

Minimax does not perform pruning, while Alpha-Beta can avoid exploring
branches that cannot affect the final decision.

Move ordering can further improve Alpha-Beta pruning by considering
promising moves earlier.

In [24]:
print("Comparison")
print("-" * 60)

print("Minimax:")
print("Best Move:", mm_move)
print("Evaluation Score:", mm_val)
print("Nodes Generated:", mm_ng)
print("Nodes Expanded:", mm_ne)
print("Execution Time:", mm_time)

print()

print("Alpha-Beta:")
print("Best Move:", ab_move)
print("Evaluation Score:", ab_val)
print("Nodes Generated:", ab_ng)
print("Nodes Expanded:", ab_ne)
print("Nodes Pruned:", ab_np)
print("Execution Time:", ab_time)

print()

print("Alpha-Beta with Move Ordering:")
print("Best Move:", ab2_move)
print("Evaluation Score:", ab2_val)
print("Nodes Generated:", ab2_ng)
print("Nodes Expanded:", ab2_ne)
print("Nodes Pruned:", ab2_np)
print("Execution Time:", ab2_time)

Comparison
------------------------------------------------------------
Minimax:
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 66
Nodes Expanded: 36
Execution Time: 0.00027179718017578125

Alpha-Beta:
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 51
Nodes Expanded: 24
Nodes Pruned: 9
Execution Time: 0.0002574920654296875

Alpha-Beta with Move Ordering:
Best Move: RIGHT
Evaluation Score: 1
Nodes Generated: 48
Nodes Expanded: 22
Nodes Pruned: 8
Execution Time: 0.0002665519714355469
